# Озвучення «Котів-вояків» у Colab (StyleTTS2, GPU)

Дозвучує розділи книги «На волю!» голосом **Марічка Штирбулова** тією самою моделлю StyleTTS2, що й на ноутбуці, тільки на GPU Colab (на T4 це в ~16 разів швидше за реальний час).

**Перед запуском:** Runtime → Change runtime type → **GPU** (T4 підійде).

**Що потрібно мати на комп'ютері (у репозиторії цих файлів немає):**
1. книга `Ganter_warriors_1_Na_volyu_b829d0_537067.fb2` (папка `books`);
2. стиль голосу `Марічка Штирбулова.pt` (папка `data/models/styletts2/voices`, ≈2 КБ).

Завантаження книги й голосу — двома окремими комірками (2а і 2б). Комірки виконуй по черзі зверху вниз. Розділи, які вже озвучені на ноутбуці (1–20), не чіпаються: тут лишилися 21–27 («Розділ 19»–«Розділ 25»).

## 1. Встановлення (≈5–10 хв, лише раз)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
ROOT = "/content/audiobooks-generation"
if not os.path.exists(ROOT):
    !git clone -q https://github.com/flonado/audiobooks-generation.git {ROOT}
!pip -q install "transformers==5.0.0" accelerate einops einops_exts munch sentencepiece ctranslate2 soundfile librosa scipy stanza pyyaml
!pip -q install "git+https://github.com/patriotyk/ukrainian-word-stress.git" "git+https://github.com/patriotyk/ipa-uk.git" "git+https://github.com/patriotyk/styletts2-inference@209352700cf703a05ff94501290095dd9072fd5f"
print("готово")

Попередження pip про «dependency conflicts» (rasterio, jax, opencv…) — це про інші пакети Colab, нашому коду вони не заважають. Перезапускати сесію **не треба**: озвучення йде окремим процесом.

## 2. Завантаження книги та голосу (два окремі кроки)

In [ ]:
# 2a. Книга: обери файл Ganter_warriors_1_Na_volyu_b829d0_537067.fb2
import os, shutil
from google.colab import files
ROOT = "/content/audiobooks-generation"
os.makedirs(f"{ROOT}/books", exist_ok=True)
up = files.upload()
for name in up:
    shutil.move(name, f"{ROOT}/books/{name}")
print("книги:", os.listdir(f"{ROOT}/books"))

In [ ]:
# 2б. Голос: обери файл «Марічка Штирбулова.pt» (окремо від книги, бо вікно іноді дозволяє лише один файл)
os.makedirs(f"{ROOT}/data/models/styletts2/voices", exist_ok=True)
up = files.upload()
for name in up:
    shutil.move(name, f"{ROOT}/data/models/styletts2/voices/{name}")
print("голоси:", os.listdir(f"{ROOT}/data/models/styletts2/voices"))

## 3. Скрипт озвучення та перелік розділів

In [ ]:
%%writefile /content/voice.py
import sys
sys.path.insert(0, "/content/audiobooks-generation/src")
from pathlib import Path
from audiobooks.audio import save_wav, unique_path
from audiobooks.books import load_book
from audiobooks.text import parse_selection, prepare_text, slugify
from audiobooks.tts.local import make_engine, synthesize_text

book, voice, out_dir, selection = sys.argv[1:5]
chapters = load_book(book)
eng = make_engine("styletts2", voice, 1.0, device="cuda")
out = Path(out_dir)
for n in parse_selection(selection, len(chapters)):
    title, body = chapters[n - 1]
    print()
    print(f"=== {n}. {title} ===", flush=True)
    pcm = synthesize_text(eng, prepare_text(title, body))
    path = unique_path(out, f"{n:02d}_{slugify(title)}_styletts2_{slugify(voice)}", "v")
    save_wav(path, pcm, eng.sample_rate)


In [ ]:
import glob
BOOK = glob.glob(f"{ROOT}/books/*.fb2")[0]
VOICE = "Марічка Штирбулова"
!cd {ROOT} && PYTHONPATH=src python -c "from audiobooks.books import load_book; [print(i, t, len(x)) for i, (t, x) in enumerate(load_book(r'{BOOK}'), 1)]"

Має бути 27 розділів. Номери в лівому стовпчику — це ті, що вказуються нижче (21 = «Розділ 19», 27 = «Розділ 25»).

## 4. Озвучення (≈10–20 хв на T4)

In [ ]:
FIRST, LAST = 21, 27          # які розділи озвучувати (за потреби зміни)
OUT = "/content/out"
os.makedirs(OUT, exist_ok=True)
missing = [n for n in range(FIRST, LAST + 1) if not glob.glob(f"{OUT}/{n:02d}_*.wav")]
print("ще треба озвучити:", missing or "нічого, усе вже є")
if missing:
    SEL = ",".join(map(str, missing))
    !python /content/voice.py "{BOOK}" "{VOICE}" "{OUT}" "{SEL}"

Якщо сесія обірвалася посеред роботи, просто запусти цю комірку ще раз: готові розділи пропускаються (файли лежать у `/content/out`; після повного скидання сесії вони зникають, тож не затягуй із наступним кроком).

## 5. Перевірка результату

In [ ]:
import wave
total = 0
for f in sorted(glob.glob(f"{OUT}/*.wav")):
    with wave.open(f) as w:
        sec = w.getnframes() / w.getframerate()
    total += sec
    print(f"{os.path.basename(f)}  {sec/60:5.1f} хв  {os.path.getsize(f)/2**20:5.1f} МБ")
print(f"разом: {total/60:.1f} хв")
from IPython.display import Audio
last = sorted(glob.glob(f"{OUT}/*.wav"))[-1]
Audio(last)

## 6. Забрати файли

Файли великі (≈35 МБ на розділ), тому надійніше **через Google Drive**. Виконай одну з двох комірок.

In [ ]:
# Варіант A: скопіювати на Google Drive (рекомендую)
from google.colab import drive
drive.mount("/content/drive")
DST = "/content/drive/MyDrive/warriors_audio"
os.makedirs(DST, exist_ok=True)
for f in glob.glob(f"{OUT}/*.wav"):
    shutil.copy(f, DST)
print("скопійовано в", DST, os.listdir(DST))

In [ ]:
# Варіант B: скачати одним архівом через браузер
shutil.make_archive("/content/warriors_audio", "zip", OUT)
files.download("/content/warriors_audio.zip")